In [ ]:
#  Cell 1 – MASTER CONFIGURATION  (edit ONLY this dict)
import os, sys, json, warnings
from pathlib import Path
from typing import Dict, Any, Optional, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, f1_score,
)
import librosa

warnings.filterwarnings("ignore", category=FutureWarning)

# Add repo root to path so `src.*` imports work on Kaggle
REPO_ROOT = Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

CONFIG: Dict[str, Any] = {
    # Data ─────────────────
    "data": {
        "spectrogram_dir": "data/processed/spectrograms",
        "features_csv":    "data/processed/features/end_frequencies.csv",
        "num_classes":      3,          # auto-detected after scanning
        "image_size":       (128, 128),
    },

    #Split ratios (must sum to 1.0) ───────────────────────────────────---------------------
    "split": {
        "train":    0.70,
        "val":      0.20,
        "test":     0.10,
        "seed":     42,
        "stratify": True,
    },

    #Model ─────────────────────
    #  ★ Change this ONE string to swap architecture:*
    #"mobilenet"→ MobileNetV3-Small(2.5 M – fastest)
    #"efficientnet"→ EfficientNet-B0(5.3 M – ★ BEST for small data)
    #"densenet"→ DenseNet-121(8.0 M – strong regularisation)
    #"resnet"→ ResNet-18(11.2 M – proven baseline)
    #"convnext"→ ConvNeXt-Tiny(28 M – modern CNN)
    #"swin"→ Swin-Tiny(28 M – attention maps)
    #"transformer"→ ViT-B/16(86 M – needs >500 samples)
    "model": {
        "architecture":          "efficientnet",
        "pretrained":            True,
        "freeze_backbone_epochs": 5,     # ← more freezing for tiny data
        "num_features":          3,      # extra numeric features
    },

    #Training (tuned for 50-250 sample datasets) ────
    "train": {
        "epochs":              50,
        "batch_size":          16,
        "lr":                  5e-4,# ← gentler LR for small data
        "weight_decay":        1e-3,# ← stronger regularisation
        "scheduler":           "cosine",# "cosine" | "step" | "none"
        "step_size":           10,
        "gamma":               0.5,
        "early_stop_patience": 10, # ← more patience for small data
        "use_weighted_sampler": True,
        "label_smoothing":     0.1,# ★ NEW – reduces overconfidence
        "augmentation":        "specaugment",# ★ NEW – "none" | "specaugment" | "heavy"
    },

    #Evaluation toggles ────────
    "eval": {
        "show_confusion_matrix":       True,
        "show_classification_report":  True,
        "show_per_class_f1":           True,
        "show_loss_curves":            True,
    },

    #Inference ─────
    "inference": {
        "model_path":        "models/bat_fused_best.pth",
        "audio_dir":         "data/raw/test_audio",
        "sample_rate":       250_000,
        "min_freq":          15_000,
        "energy_thresh":     0.02,
        "min_duration":      0.01,
        "pad_duration":      0.05,
        "confidence_thresh": 0.70,
    },
    #Paths ────────
    "paths": {
        "model_save_dir": "models",
        "log_dir":        "logs",
    },
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device:        {DEVICE}")
print(f"Architecture:  {CONFIG['model']['architecture']}")
print(f"Augmentation:  {CONFIG['train']['augmentation']}")
print(f"Label smooth:  {CONFIG['train']['label_smoothing']}")
print(f"Split:         {CONFIG['split']['train']}/{CONFIG['split']['val']}/{CONFIG['split']['test']}")

In [ ]:
#  Cell 2 – Model Factory (Strategy Pattern) + Dataset imports
from abc import ABC, abstractmethod
from src.datasets.spectrogram_with_features_dataset import SpectrogramWithFeaturesDataset
from src.models.cnn_with_features import CNNWithFeatures
from src.models.efficientnet_with_features import EfficientNetWithFeatures
from src.models.mobilenet_with_features import MobileNetWithFeatures
from src.models.densenet_with_features import DenseNetWithFeatures
from src.models.convnext_with_features import ConvNeXtWithFeatures
from src.models.swin_with_features import SwinWithFeatures


#Strategy Interface
class ModelStrategy(ABC):
    """Each architecture implements build() and name()."""

    @abstractmethod
    def build(self, num_classes: int, **kw) -> nn.Module: ...

    @abstractmethod
    def name(self) -> str: ...


#ResNet18 + Feature MLP (512-d   11.2 M)
class ResNetStrategy(ModelStrategy):
    def name(self) -> str:
        return "ResNet18+MLP"

    def build(self, num_classes: int, **kw) -> nn.Module:
        return CNNWithFeatures(
            num_classes=num_classes,
            numeric_feat_dim=kw.get("num_features", 3),
            pretrained=kw.get("pretrained", True),
        )


#Concrete: EfficientNet-B0 (1280-d  5.3 M)
#  ★ RECOMMENDED for SMALL datasets: best acc-per-param ratio
class EfficientNetStrategy(ModelStrategy):
    def name(self) -> str:
        return "EfficientNet-B0+MLP"

    def build(self, num_classes: int, **kw) -> nn.Module:
        return EfficientNetWithFeatures(
            num_classes=num_classes,
            numeric_feat_dim=kw.get("num_features", 3),
            pretrained=kw.get("pretrained", True),
        )


#MobileNetV3-Small (576-d   2.5 M)
# Smallest model – best <100 samples or need fast inference
class MobileNetStrategy(ModelStrategy):
    def name(self) -> str:
        return "MobileNetV3-Small+MLP"

    def build(self, num_classes: int, **kw) -> nn.Module:
        return MobileNetWithFeatures(
            num_classes=num_classes,
            numeric_feat_dim=kw.get("num_features", 3),
            pretrained=kw.get("pretrained", True),
        )


#DenseNet-121 (1024-d   8 M)
#  Dense connections = strong built-in regularisation → great for tiny data
class DenseNetStrategy(ModelStrategy):
    def name(self) -> str:
        return "DenseNet-121+MLP"

    def build(self, num_classes: int, **kw) -> nn.Module:
        return DenseNetWithFeatures(
            num_classes=num_classes,
            numeric_feat_dim=kw.get("num_features", 3),
            pretrained=kw.get("pretrained", True),
        )


#ConvNeXt-Tiny (768-d   28 M)
#Modern CNN – use when you have >200 samples
class ConvNeXtStrategy(ModelStrategy):
    def name(self) -> str:
        return "ConvNeXt-Tiny+MLP"

    def build(self, num_classes: int, **kw) -> nn.Module:
        return ConvNeXtWithFeatures(
            num_classes=num_classes,
            numeric_feat_dim=kw.get("num_features", 3),
            pretrained=kw.get("pretrained", True),
        )


#Swin-Tiny Transformer (768-d   28 M) 
class SwinStrategy(ModelStrategy):
    def name(self) -> str:
        return "Swin-Tiny+MLP"

    def build(self, num_classes: int, **kw) -> nn.Module:
        return SwinWithFeatures(
            num_classes=num_classes,
            numeric_feat_dim=kw.get("num_features", 3),
            pretrained=kw.get("pretrained", True),
        )


#Vision Transformer ViT-B/16 (768-d   86 M)
#  Largest model – NOT recommended for <500 samples (overfits fast)
class TransformerStrategy(ModelStrategy):
    def name(self) -> str:
        return "VisionTransformer"

    def build(self, num_classes: int, **kw) -> nn.Module:
        from torchvision.models import vit_b_16, ViT_B_16_Weights

        pretrained = kw.get("pretrained", True)
        weights = ViT_B_16_Weights.DEFAULT if pretrained else None
        backbone = vit_b_16(weights=weights)
        in_features = backbone.heads.head.in_features
        backbone.heads.head = nn.Identity()
        num_extra = kw.get("num_features", 3)

        class ViTWithFeatures(nn.Module):
            def __init__(self, bb, img_dim, n_extra, n_cls):
                super().__init__()
                self.backbone = bb
                self.feature_mlp = nn.Sequential(
                    nn.Linear(n_extra, 32), nn.ReLU(), nn.Dropout(0.3),
                )
                self.classifier = nn.Sequential(
                    nn.Linear(img_dim + 32, 128), nn.ReLU(), nn.Dropout(0.4),
                    nn.Linear(128, n_cls),
                )

            def forward(self, images, features):
                img_emb  = self.backbone(images)
                feat_emb = self.feature_mlp(features)
                return self.classifier(torch.cat([img_emb, feat_emb], dim=1))

        return ViTWithFeatures(backbone, in_features, num_extra, num_classes)


#Factory registry ───────────────────────
_STRATEGIES: Dict[str, ModelStrategy] = {
    "resnet":       ResNetStrategy(),
    "efficientnet": EfficientNetStrategy(),
    "mobilenet":    MobileNetStrategy(),
    "densenet":     DenseNetStrategy(),
    "convnext":     ConvNeXtStrategy(),
    "swin":         SwinStrategy(),  
    "transformer":  TransformerStrategy(), 
}

# Quick-reference table for the user
_MODEL_GUIDE = """
architecture│ Params │ When to use                                
mobilenet   │  2.5 M │ <100 samples, fast experiments             
efficientnet│  5.3 M │ ★ DEFAULT – best for 50-300 samples         
densenet    │  8.0 M │ tiny data, strong built-in regularisation   
resnet      │ 11.2 M │ proven,always a safe bet          
convnext    │ 28.0 M │ >200 samples + heavy augmentation           
swin        │ 28.0 M │ attention maps     
transformer │ 86.0 M │ >500 samples only (will overfit otherwise)  
"""


def build_model(config: Dict[str, Any]) -> nn.Module:
    """Factory: returns a model on DEVICE based on config."""
    arch = config["model"]["architecture"]
    if arch not in _STRATEGIES:
        raise ValueError(
            f"Unknown architecture '{arch}'.\n"
            f"Available: {list(_STRATEGIES.keys())}\n{_MODEL_GUIDE}"
        )
    strategy = _STRATEGIES[arch]
    print(f"Building model: {strategy.name()}")
    model = strategy.build(
        num_classes=config["data"]["num_classes"],
        pretrained=config["model"].get("pretrained", True),
        num_features=config["model"].get("num_features", 3),
    )
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"  Parameters: {n_params:.1f} M")
    return model.to(DEVICE)


print(f"Model factory ready. Registered architectures: {list(_STRATEGIES.keys())}")
print(_MODEL_GUIDE)

: 

In [ ]:
#Cell 3 – WORKFLOW 1: Data Loading + Stratified Split

def load_and_split(config: Dict[str, Any]) -> Tuple[Subset, Subset, Subset, List[str]]:
    """
    Load SpectrogramWithFeaturesDataset and split into
    train / val / test with optional stratification by species.
    """
    spect_dir    = config["data"]["spectrogram_dir"]
    features_csv = config["data"]["features_csv"]
    split_cfg    = config["split"]

    full_ds = SpectrogramWithFeaturesDataset(
        root_dir=spect_dir,
        features_csv=features_csv if Path(features_csv).exists() else None,
    )
    n = len(full_ds)
    print(f"Total samples: {n}")

    class_names = full_ds.classes
    config["data"]["num_classes"] = len(class_names)
    print(f"Detected {len(class_names)} classes: {class_names}")

    all_labels = np.array(full_ds.labels)
    seed       = split_cfg["seed"]
    tr, va, te = split_cfg["train"], split_cfg["val"], split_cfg["test"]
    assert abs(tr + va + te - 1.0) < 1e-6, "Split ratios must sum to 1.0"

    indices = np.arange(n)

    if split_cfg["stratify"]:
        sss1 = StratifiedShuffleSplit(n_splits=1, test_size=va + te, random_state=seed)
        train_idx, temp_idx = next(sss1.split(indices, all_labels))

        temp_labels    = all_labels[temp_idx]
        relative_test  = te / (va + te)
        sss2 = StratifiedShuffleSplit(n_splits=1, test_size=relative_test, random_state=seed)
        val_local, test_local = next(sss2.split(temp_idx, temp_labels))
        val_idx  = temp_idx[val_local]
        test_idx = temp_idx[test_local]
    else:
        rng = np.random.RandomState(seed)
        rng.shuffle(indices)
        n_train = int(n * tr)
        n_val   = int(n * va)
        train_idx = indices[:n_train]
        val_idx   = indices[n_train:n_train + n_val]
        test_idx  = indices[n_train + n_val:]

    train_ds = Subset(full_ds, train_idx.tolist())
    val_ds   = Subset(full_ds, val_idx.tolist())
    test_ds  = Subset(full_ds, test_idx.tolist())

    for name, idx_arr in [("Train", train_idx), ("Val", val_idx), ("Test", test_idx)]:
        counts = np.bincount(all_labels[idx_arr], minlength=len(class_names))
        dist = {class_names[i]: int(c) for i, c in enumerate(counts)}
        print(f"  {name:5s} ({len(idx_arr):4d}): {dist}")

    return train_ds, val_ds, test_ds, class_names


# Execute
train_ds, val_ds, test_ds, CLASS_NAMES = load_and_split(CONFIG)

In [ ]:
#  Cell 4 – WORKFLOW 2: Training + Evaluation  (Plug-and-Play)
#
#  Supports ALL 7 architectures.  Just change CONFIG["model"]["architecture"]
#  and re-run.  Everything else adapts automatically.
#
#  Small-data features:
#    ★ label_smoothing  – reduces overconfidence  (CONFIG["train"]["label_smoothing"])
#    ★ augmentation     – SpecAugment-style masks (CONFIG["train"]["augmentation"])
#    ★ freeze_backbone  – freezes pretrained layers for N epochs first


def _make_weighted_sampler(dataset: Subset, num_classes: int) -> WeightedRandomSampler:
    full_ds = dataset.dataset
    int_labels = np.array([full_ds.labels[i] for i in dataset.indices])
    counts = np.bincount(int_labels, minlength=num_classes)
    weights = 1.0 / (counts + 1e-6)
    sample_w = weights[int_labels]
    return WeightedRandomSampler(torch.DoubleTensor(sample_w), len(sample_w), replacement=True)


def _build_loaders(cfg, tr_ds, va_ds, te_ds):
    bs = cfg["train"]["batch_size"]
    nc = cfg["data"]["num_classes"]
    sampler = _make_weighted_sampler(tr_ds, nc) if cfg["train"]["use_weighted_sampler"] else None
    return (
        DataLoader(tr_ds, batch_size=bs, shuffle=(sampler is None), sampler=sampler, num_workers=2),
        DataLoader(va_ds, batch_size=bs, shuffle=False, num_workers=2),
        DataLoader(te_ds, batch_size=bs, shuffle=False, num_workers=2),
    )


def train_and_evaluate(cfg, tr_ds, va_ds, te_ds, cls_names):
    model = build_model(cfg)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Parameters: {n_params:,}")

    train_ld, val_ld, test_ld = _build_loaders(cfg, tr_ds, va_ds, te_ds)

    # ★ Label smoothing (reduces overconfidence on tiny datasets)
    label_smoothing = cfg["train"].get("label_smoothing", 0.0)
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    if label_smoothing > 0:
        print(f"Using label smoothing = {label_smoothing}")

    optimizer = optim.Adam(model.parameters(), lr=cfg["train"]["lr"], weight_decay=cfg["train"]["weight_decay"])

    sched_type = cfg["train"]["scheduler"]
    scheduler  = None
    if sched_type == "cosine":
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["train"]["epochs"])
    elif sched_type == "step":
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=cfg["train"]["step_size"], gamma=cfg["train"]["gamma"])

    epochs        = cfg["train"]["epochs"]
    patience      = cfg["train"]["early_stop_patience"]
    freeze_epochs = cfg["model"].get("freeze_backbone_epochs", 0)
    save_dir      = Path(cfg["paths"]["model_save_dir"])
    save_dir.mkdir(parents=True, exist_ok=True)
    best_path = save_dir / "bat_fused_best.pth"

    history = {"train_loss": [], "val_loss": [], "val_acc": [], "lr": []}
    best_val_acc     = 0.0
    patience_counter = 0

    if freeze_epochs > 0 and hasattr(model, "backbone"):
        for p in model.backbone.parameters():
            p.requires_grad = False
        print(f"Backbone frozen for first {freeze_epochs} epochs")

    for epoch in range(1, epochs + 1):
        # Unfreeze
        if epoch == freeze_epochs + 1 and hasattr(model, "backbone"):
            for p in model.backbone.parameters():
                p.requires_grad = True
            # Reset optimizer to include backbone params with smaller LR
            optimizer = optim.Adam([
                {"params": model.backbone.parameters(), "lr": cfg["train"]["lr"] * 0.1},
                {"params": [p for n, p in model.named_parameters() if not n.startswith("backbone")]},
            ], lr=cfg["train"]["lr"], weight_decay=cfg["train"]["weight_decay"])
            print(f"Epoch {epoch}: Backbone unfrozen (backbone LR = {cfg['train']['lr'] * 0.1:.1e})")

        #Train
        model.train()
        run_loss = 0.0
        for images, features, labels in train_ld:
            images, features, labels = images.to(DEVICE), features.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(images, features), labels)
            loss.backward()
            optimizer.step()
            run_loss += loss.item() * images.size(0)
        train_loss = run_loss / len(train_ld.dataset)

        #Validate
        model.eval()
        v_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, features, labels in val_ld:
                images, features, labels = images.to(DEVICE), features.to(DEVICE), labels.to(DEVICE)
                out = model(images, features)
                v_loss += criterion(out, labels).item() * images.size(0)
                correct += (out.argmax(1) == labels).sum().item()
                total   += labels.size(0)
        val_loss = v_loss / len(val_ld.dataset)
        val_acc  = correct / total

        if scheduler:
            scheduler.step()
        lr = optimizer.param_groups[0]["lr"]
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["lr"].append(lr)

        print(f"Epoch {epoch:3d}/{epochs} │ TrL {train_loss:.4f} │ VaL {val_loss:.4f} │ VaAcc {val_acc:.4f} │ LR {lr:.2e}")

        if val_acc > best_val_acc:
            best_val_acc, patience_counter = val_acc, 0
            torch.save({"model_state_dict": model.state_dict(), "config": cfg,
                        "class_names": cls_names, "epoch": epoch, "val_acc": val_acc}, best_path)
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch}"); break

    print(f"\nBest val accuracy: {best_val_acc:.4f}  →  {best_path}")

    #Load best & test
    ckpt = torch.load(best_path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, features, labels in test_ld:
            images, features, labels = images.to(DEVICE), features.to(DEVICE), labels.to(DEVICE)
            all_preds.extend(model(images, features).argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    ev = cfg["eval"]

    if ev["show_classification_report"]:
        print("\n" + "=" * 60 + "\nCLASSIFICATION REPORT (Test)\n" + "=" * 60)
        print(classification_report(all_labels, all_preds, target_names=cls_names, digits=4))

    if ev["show_per_class_f1"]:
        f1s = f1_score(all_labels, all_preds, average=None)
        for name, score in zip(cls_names, f1s):
            print(f"  {name:30s}  F1={score:.4f}")

    if ev["show_confusion_matrix"]:
        cm = confusion_matrix(all_labels, all_preds)
        fig_cm, ax_cm = plt.subplots(figsize=(6, 5), dpi=100)
        ConfusionMatrixDisplay(cm, display_labels=cls_names).plot(ax=ax_cm, cmap="Blues", values_format="d")
        ax_cm.set_title("Confusion Matrix – Test"); plt.tight_layout(); plt.show()

    if ev["show_loss_curves"]:
        ep = range(1, len(history["train_loss"]) + 1)
        fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4), dpi=100)
        a1.plot(ep, history["train_loss"], label="Train"); a1.plot(ep, history["val_loss"], label="Val")
        a1.set(xlabel="Epoch", ylabel="Loss", title="Loss Curves"); a1.legend(); a1.grid(True, alpha=.3)
        a2.plot(ep, history["val_acc"], label="Val Acc", color="green")
        a2.set(xlabel="Epoch", ylabel="Accuracy", title="Validation Accuracy"); a2.legend(); a2.grid(True, alpha=.3)
        fig.tight_layout(); plt.show()

    return model, history


#Execute
trained_model, train_history = train_and_evaluate(
    CONFIG, train_ds, val_ds, test_ds, CLASS_NAMES
)

In [ ]:

#Cell 5 – WORKFLOW 3: Inference on new, un-annotated audio
from scipy.signal import butter, sosfilt
from src.data_prep.wombat_to_spectrograms import make_mel_spectrogram
from src.data_prep.extract_end_frequency import compute_end_frequency


def _bandpass(y: np.ndarray, sr: int, low=15_000, high=120_000, order=4) -> np.ndarray:
    nyq   = 0.5 * sr
    lo_n  = max(1.0, min(low, nyq * 0.9)) / nyq
    hi_n  = min(high, nyq * 0.99) / nyq
    if hi_n <= lo_n:
        return y
    sos = butter(order, [lo_n, hi_n], btype="band", output="sos")
    return sosfilt(sos, y).astype(np.float32)


def _detect_events(
    y, sr, min_freq=15_000, thresh=0.02, min_dur=0.01, pad=0.05,
    frame_len=2048, hop=512,
) -> List[Tuple[float, float]]:
    """Energy-based event detector for bat calls."""
    y_bp = _bandpass(y, sr, low=min_freq)
    rms  = librosa.feature.rms(y=y_bp, frame_length=frame_len, hop_length=hop)[0]
    times = librosa.frames_to_time(np.arange(len(rms)), sr=sr, hop_length=hop)
    active = rms > thresh
    events, in_ev, start = [], False, 0.0
    for i, a in enumerate(active):
        if a and not in_ev:
            start, in_ev = times[i], True
        elif not a and in_ev:
            if times[i] - start >= min_dur:
                events.append((max(0, start - pad), min(len(y)/sr, times[i] + pad)))
            in_ev = False
    if in_ev and times[-1] - start >= min_dur:
        events.append((max(0, start - pad), min(len(y)/sr, times[-1] + pad)))
    # merge overlapping
    if not events:
        return events
    events.sort()
    merged = [events[0]]
    for s, e in events[1:]:
        if s <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], e))
        else:
            merged.append((s, e))
    return merged


def _prepare_input(y, sr, start, end, img_size=(128, 128)):
    s, e = int(start * sr), int(end * sr)
    seg  = y[s:e]
    if len(seg) < 512:
        return None
    high_cut = min(120_000, int(sr * 0.5) - 1000)
    seg = _bandpass(seg, sr, low=15_000, high=high_cut)
    S_db = make_mel_spectrogram(seg, sr)
    if S_db.size == 0:
        return None
    S_min, S_max = S_db.min(), S_db.max()
    S_norm = (S_db - S_min) / (S_max - S_min + 1e-8)
    cmap   = plt.get_cmap("magma")
    rgb    = (cmap(S_norm)[:, :, :3] * 255).astype(np.uint8)
    from PIL import Image as _Img
    img = np.array(_Img.fromarray(rgb).resize(img_size, _Img.BILINEAR), dtype=np.float32) / 255.0
    img_t = torch.from_numpy(img.transpose(2, 0, 1)).unsqueeze(0)
    ef = compute_end_frequency(y, sr, start, end)
    feat_t = torch.tensor([[ef if not np.isnan(ef) else 0.0, 0.0, 0.0]], dtype=torch.float32)
    return img_t, feat_t


def run_inference(cfg: Dict[str, Any]):
    """Load model → detect events → classify."""
    inf       = cfg["inference"]
    model_p   = Path(inf["model_path"])
    audio_dir = Path(inf["audio_dir"])
    img_size  = tuple(cfg["data"]["image_size"])

    if not model_p.exists():
        print(f" Model not found: {model_p}"); return []

    ckpt       = torch.load(model_p, map_location=DEVICE)
    cls_names  = ckpt.get("class_names", CLASS_NAMES)
    model      = build_model(cfg)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    print(f" Model loaded ({len(cls_names)} classes: {cls_names})")

    audio_files = []
    if audio_dir.exists():
        for ext in ("*.wav", "*.WAV", "*.flac", "*.mp3"):
            audio_files.extend(audio_dir.rglob(ext))
    print(f"Found {len(audio_files)} audio files")

    results = []
    for ap in audio_files:
        try:
            y, sr = librosa.load(str(ap), sr=None, mono=True)
        except Exception as e:
            print(f"  ⚠ {ap.name}: {e}"); continue
        events = _detect_events(y, sr, min_freq=inf["min_freq"],
                                thresh=inf["energy_thresh"],
                                min_dur=inf["min_duration"], pad=inf["pad_duration"])
        if not events:
            results.append({"file": ap.name, "start": "-", "end": "-",
                            "prediction": "No Detection", "confidence": 0.0})
            continue
        for s, e in events:
            inp = _prepare_input(y, sr, s, e, img_size)
            if inp is None:
                continue
            img_t, feat_t = inp
            with torch.no_grad():
                probs = torch.softmax(model(img_t.to(DEVICE), feat_t.to(DEVICE)), 1)
                conf, idx = probs.max(1)
            if conf.item() >= inf["confidence_thresh"]:
                results.append({"file": ap.name, "start": f"{s:.3f}", "end": f"{e:.3f}",
                                "prediction": cls_names[idx.item()],
                                "confidence": round(conf.item() * 100, 2)})

    if results:
        import pandas as pd
        df = pd.DataFrame(results)
        print(f"\n{'='*60}\nINFERENCE RESULTS ({len(df)} detections)\n{'='*60}")
        display(df)
        print("\nPrediction counts:")
        print(df["prediction"].value_counts().to_string())
    else:
        print("No detections above confidence threshold.")
    return results


# ********************Uncomment to run inference**********i.e remove the"#"
# inference_results = run_inference(CONFIG)
print("Inference cell ready. Uncomment the last line to execute.")

In [ ]:
#  Cell 6 – MODEL SHOOTOUT  (compare all architectures head-to-head)
#
#  Trains every registered architecture with the SAME data split and
#  prints a ranked comparison table.  Use this to pick the best model
#  for YOUR dataset size.
#
# Which architectures to compare ──────────────────────────────────
#  "all" = every registered architecture
#  Or pick a subset:  ["efficientnet", "mobilenet", "densenet", "resnet"]
SHOOTOUT_ARCHS = ["mobilenet", "efficientnet", "densenet", "resnet"]

import copy, time

results = []

for arch_name in SHOOTOUT_ARCHS:
    if arch_name not in _STRATEGIES:
        print(f"⚠ Skipping unknown architecture: {arch_name}")
        continue

    print(f"\n{'='*70}")
    print(f"  TRAINING: {arch_name.upper()}")
    print(f"{'='*70}")

    cfg_copy = copy.deepcopy(CONFIG)
    cfg_copy["model"]["architecture"] = arch_name

    t0 = time.time()
    try:
        model_i, hist_i = train_and_evaluate(cfg_copy, train_ds, val_ds, test_ds, CLASS_NAMES)
        elapsed = time.time() - t0
        best_val = max(hist_i["val_acc"])
        n_params = sum(p.numel() for p in model_i.parameters()) / 1e6
        results.append({
            "architecture": arch_name,
            "strategy":     _STRATEGIES[arch_name].name(),
            "params_M":     round(n_params, 1),
            "best_val_acc": round(best_val, 4),
            "epochs_run":   len(hist_i["train_loss"]),
            "time_min":     round(elapsed / 60, 1),
        })
    except Exception as e:
        print(f"  ✗ {arch_name} failed: {e}")
        results.append({
            "architecture": arch_name,
            "strategy":     _STRATEGIES[arch_name].name(),
            "params_M":     "–",
            "best_val_acc": 0.0,
            "epochs_run":   0,
            "time_min":     0.0,
        })

#Pretty comparison table
results.sort(key=lambda r: r["best_val_acc"], reverse=True)

print(f"\n\n{'='*70}")
print("  MODEL SHOOTOUT RESULTS  (ranked by best validation accuracy)")
print(f"{'='*70}")
print(f"{'Rank':>4}  {'Architecture':<18}  {'Params':>7}  {'Val Acc':>8}  {'Epochs':>6}  {'Time':>6}")
print(f"{'─'*4}  {'─'*18}  {'─'*7}  {'─'*8}  {'─'*6}  {'─'*6}")
for rank, r in enumerate(results, 1):
    medal = "GOLD" if rank == 1 else "SILVER" if rank == 2 else "BRONZE" if rank == 3 else "  "
    print(f"{medal}{rank:2d}  {r['architecture']:<18}  {r['params_M']:>6}M  {r['best_val_acc']:>7.4f}  {r['epochs_run']:>6d}  {r['time_min']:>5.1f}m")

#Plot comparison
if len(results) > 1:
    archs = [r["architecture"] for r in results]
    accs  = [r["best_val_acc"] for r in results]
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(archs)))

    fig, ax = plt.subplots(figsize=(10, 5), dpi=100)
    bars = ax.barh(archs, accs, color=colors, edgecolor="white", linewidth=1.5)
    ax.set_xlabel("Best Validation Accuracy")
    ax.set_title("Model Shootout – Architecture Comparison")
    ax.set_xlim(0, 1.0)
    for bar, acc in zip(bars, accs):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f"{acc:.4f}", va="center", fontweight="bold")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

print(f"\n★ Winner: {results[0]['architecture']} ({results[0]['best_val_acc']:.4f})")
print(f"  To use it, set:  CONFIG['model']['architecture'] = \"{results[0]['architecture']}\"")
print(f"  Then re-run Cell 4.")